In [12]:
from dataclasses import dataclass,field
from datetime import datetime,timezone
from typing import Optional
import pandas as pd




In [13]:
@dataclass
class Transaction:
    transaction_id: str
    card_id: str
    amount: float
    currency: str

    merchant_id: str
    mcc: str
    country: str

    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    channel: str = "card_present"

    latitude: Optional[float] = None
    longitude: Optional[float] = None
    card_country: Optional[str] = None
        
        

In [14]:
txn = Transaction(
    transaction_id="t001",
    card_id="card_123abc",
    amount=50.00,
    currency="USD",
    merchant_id="m_starbucks_042",
    mcc="5812",
    country="US",
)

txn


Transaction(transaction_id='t001', card_id='card_123abc', amount=50.0, currency='USD', merchant_id='m_starbucks_042', mcc='5812', country='US', timestamp=datetime.datetime(2026, 9, 29, 5, 31, 48, 953489, tzinfo=datetime.timezone.utc), channel='card_present', latitude=None, longitude=None, card_country=None)

In [15]:
def check_amount_threshold(txn, threshold=2000.0):
    triggered = txn.amount > threshold
    reason = f"amount ${txn.amount:.2f} exceeds threshold ${threshold:.2f}" if triggered else ""
    return {
        "rule_name": "amount_threshold",
        "triggered": triggered,
        "score": 2.0 if triggered else 0.0,
        "reason": reason,
    }


In [16]:
result = check_amount_threshold(txn)
result

{'rule_name': 'amount_threshold',
 'triggered': False,
 'score': 0.0,
 'reason': ''}

In [17]:
big_txn = Transaction(
    transaction_id="t002",
    card_id="card_123abc",
    amount=3500.00,
    currency="USD",
    merchant_id="m_electronics_01",
    mcc="5732",
    country="US",
)

check_amount_threshold(big_txn)

{'rule_name': 'amount_threshold',
 'triggered': True,
 'score': 2.0,
 'reason': 'amount $3500.00 exceeds threshold $2000.00'}

In [18]:
from collections import defaultdict

transaction_history = defaultdict(list)  # card_id -> list of past Transactions

In [19]:
from datetime import timedelta

def check_velocity(txn, history, window_seconds=600, max_count=5):
    window = timedelta(seconds=window_seconds)
    cutoff = txn.timestamp - window

    recent = [t for t in history[txn.card_id] if t.timestamp >= cutoff]
    count = len(recent) + 1  # +1 includes the current transaction

    triggered = count > max_count
    reason = f"{count} txns in {window_seconds}s (limit {max_count})" if triggered else ""

    return {
        "rule_name": "velocity",
        "triggered": triggered,
        "score": 2.0 if triggered else 0.0,
        "reason": reason,
    }

In [20]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc)

for i in range(6):
    t = Transaction(
        transaction_id=f"v{i}",
        card_id="card_velocity_test",
        amount=20.00,
        currency="USD",
        merchant_id=f"m_{i}",
        mcc="5812",
        country="US",
        timestamp=now + timedelta(seconds=i * 30),  # 30 seconds apart
    )
    result = check_velocity(t, transaction_history)
    print(f"txn {i}: triggered={result['triggered']}  {result['reason']}")
    transaction_history[t.card_id].append(t)  # record AFTER checking
    

txn 0: triggered=False  
txn 1: triggered=False  
txn 2: triggered=False  
txn 3: triggered=False  
txn 4: triggered=False  
txn 5: triggered=True  6 txns in 600s (limit 5)


In [21]:
def run_engine(txn, history, rules=None, review_threshold=3.0, decline_threshold=6.0):
    if rules is None:
        rules = [check_amount_threshold, check_velocity]

    results = []
    for rule_fn in rules:
        if rule_fn is check_velocity:
            result = rule_fn(txn, history)
        else:
            result = rule_fn(txn)
        results.append(result)

    total_score = sum(r["score"] for r in results if r["triggered"])

    if total_score >= decline_threshold:
        decision = "DECLINE"
    elif total_score >= review_threshold:
        decision = "REVIEW"
    else:
        decision = "APPROVE"

    history[txn.card_id].append(txn)  # record after evaluating, not before

    return {
        "transaction_id": txn.transaction_id,
        "total_score": total_score,
        "decision": decision,
        "triggered_rules": [r for r in results if r["triggered"]],
    }


In [22]:
def check_amount_threshold(txn, history, threshold=2000.0):
    triggered = txn.amount > threshold
    reason = f"amount ${txn.amount:.2f} exceeds threshold ${threshold:.2f}" if triggered else ""
    return {
        "rule_name": "amount_threshold",
        "triggered": triggered,
        "score": 2.0 if triggered else 0.0,
        "reason": reason,
    }

In [23]:
def run_engine(txn, history, rules=None, review_threshold=3.0, decline_threshold=6.0):
    if rules is None:
        rules = [check_amount_threshold, check_velocity]

    results = [rule_fn(txn, history) for rule_fn in rules]

    total_score = sum(r["score"] for r in results if r["triggered"])

    if total_score >= decline_threshold:
        decision = "DECLINE"
    elif total_score >= review_threshold:
        decision = "REVIEW"
    else:
        decision = "APPROVE"

    history[txn.card_id].append(txn)

    return {
        "transaction_id": txn.transaction_id,
        "total_score": total_score,
        "decision": decision,
        "triggered_rules": [r for r in results if r["triggered"]],
    }

In [24]:
result = run_engine(big_txn, transaction_history)
result


{'transaction_id': 't002',
 'total_score': 2.0,
 'decision': 'APPROVE',
 'triggered_rules': [{'rule_name': 'amount_threshold',
   'triggered': True,
   'score': 2.0,
   'reason': 'amount $3500.00 exceeds threshold $2000.00'}]}

In [25]:
now2 = datetime.now(timezone.utc)

# fire off 6 rapid small transactions to build up velocity history
for i in range(5):
    t = Transaction(
        transaction_id=f"stack{i}",
        card_id="card_stack_test",
        amount=10.00,
        currency="USD",
        merchant_id=f"m_{i}",
        mcc="5812",
        country="US",
        timestamp=now2 + timedelta(seconds=i * 30),
    )
    run_engine(t, transaction_history)

# the 6th transaction is also large -> both rules should fire
final_txn = Transaction(
    transaction_id="stack5",
    card_id="card_stack_test",
    amount=2500.00,
    currency="USD",
    merchant_id="m_final",
    mcc="5812",
    country="US",
    timestamp=now2 + timedelta(seconds=150),
)
run_engine(final_txn, transaction_history)

{'transaction_id': 'stack5',
 'total_score': 4.0,
 'decision': 'REVIEW',
 'triggered_rules': [{'rule_name': 'amount_threshold',
   'triggered': True,
   'score': 2.0,
   'reason': 'amount $2500.00 exceeds threshold $2000.00'},
  {'rule_name': 'velocity',
   'triggered': True,
   'score': 2.0,
   'reason': '6 txns in 600s (limit 5)'}]}

In [26]:
def check_high_risk_mcc(txn, history, high_risk_mccs=None):
    if high_risk_mccs is None:
        high_risk_mccs = {"7995", "6051", "4829"}  # gambling, crypto/quasi-cash, wire transfer

    triggered = txn.mcc in high_risk_mccs
    reason = f"MCC {txn.mcc} is on the high-risk list" if triggered else ""

    return {
        "rule_name": "high_risk_mcc",
        "triggered": triggered,
        "score": 1.5 if triggered else 0.0,
        "reason": reason,
    }

In [27]:
result = run_engine(final_txn, transaction_history, rules=[check_amount_threshold, check_velocity, check_high_risk_mcc])
result

{'transaction_id': 'stack5',
 'total_score': 4.0,
 'decision': 'REVIEW',
 'triggered_rules': [{'rule_name': 'amount_threshold',
   'triggered': True,
   'score': 2.0,
   'reason': 'amount $2500.00 exceeds threshold $2000.00'},
  {'rule_name': 'velocity',
   'triggered': True,
   'score': 2.0,
   'reason': '7 txns in 600s (limit 5)'}]}

In [28]:
gambling_txn = Transaction(
    transaction_id="g001",
    card_id="card_gambling_test",
    amount=100.00,
    currency="USD",
    merchant_id="m_casino",
    mcc="7995",
    country="US",
)

run_engine(gambling_txn, transaction_history, rules=[check_amount_threshold, check_velocity, check_high_risk_mcc])

{'transaction_id': 'g001',
 'total_score': 1.5,
 'decision': 'APPROVE',
 'triggered_rules': [{'rule_name': 'high_risk_mcc',
   'triggered': True,
   'score': 1.5,
   'reason': 'MCC 7995 is on the high-risk list'}]}

In [29]:
import pandas as pd


In [30]:
df = pd.read_csv(r'C:\Users\Apple\Documents\HCL project\train_transaction.csv', nrows=5000)

In [31]:
df.shape

(5000, 394)

In [32]:
df.describe


<bound method NDFrame.describe of       TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0           2987000        0          86400           68.50         W  13926   
1           2987001        0          86401           29.00         W   2755   
2           2987002        0          86469           59.00         W   4663   
3           2987003        0          86499           50.00         W  18132   
4           2987004        0          86506           50.00         H   4497   
...             ...      ...            ...             ...       ...    ...   
4995        2991995        0         171600           30.00         S  16659   
4996        2991996        0         171613          274.95         W  14318   
4997        2991997        0         171629           47.95         W   1039   
4998        2991998        0         171634           40.00         W  15521   
4999        2991999        0         171642           87.00         W   6358   

     

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 15.0+ MB


In [34]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
df.columns.tolist()

['TransactionID',
 'isFraud',
 'TransactionDT',
 'TransactionAmt',
 'ProductCD',
 'card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'dist1',
 'dist2',
 'P_emaildomain',
 'R_emaildomain',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'D7',
 'D8',
 'D9',
 'D10',
 'D11',
 'D12',
 'D13',
 'D14',
 'D15',
 'M1',
 'M2',
 'M3',
 'M4',
 'M5',
 'M6',
 'M7',
 'M8',
 'M9',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'V29',
 'V30',
 'V31',
 'V32',
 'V33',
 'V34',
 'V35',
 'V36',
 'V37',
 'V38',
 'V39',
 'V40',
 'V41',
 'V42',
 'V43',
 'V44',
 'V45',
 'V46',
 'V47',
 'V48',
 'V49',
 'V50',
 'V51',
 'V52',
 'V53',
 'V54',
 'V55',
 'V56',
 'V57',
 'V58',
 'V59',
 'V60',
 'V61',
 'V62',
 'V63',
 'V64',
 'V

In [36]:
from datetime import datetime, timedelta, timezone

REFERENCE_START = datetime(2017, 12, 1, tzinfo=timezone.utc)  # arbitrary anchor, IEEE-CIS convention

def map_row_to_transaction(row):
    return Transaction(
        transaction_id=str(row['TransactionID']),
        card_id=str(row['card1']),
        amount=float(row['TransactionAmt']),
        currency="USD",
        merchant_id=str(row['ProductCD']),
        mcc="0000",  # unknown / not present in this dataset
        country="US",  # unknown / not present, using a placeholder
        timestamp=REFERENCE_START + timedelta(seconds=int(row['TransactionDT'])),
        channel="ecommerce",
    )


In [37]:
row = df.iloc[0]
txn = map_row_to_transaction(row)
txn

Transaction(transaction_id='2987000', card_id='13926', amount=68.5, currency='USD', merchant_id='W', mcc='0000', country='US', timestamp=datetime.datetime(2017, 12, 2, 0, 0, tzinfo=datetime.timezone.utc), channel='ecommerce', latitude=None, longitude=None, card_country=None)

In [38]:
run_engine(txn, transaction_history)

{'transaction_id': '2987000',
 'total_score': 0,
 'decision': 'APPROVE',
 'triggered_rules': []}

In [39]:
transaction_history = defaultdict(list)  # fresh history for this full run

In [40]:
df_sorted = df.sort_values('TransactionDT')

results = []
for _, row in df_sorted.iterrows():
    txn = map_row_to_transaction(row)
    eval_result = run_engine(txn, transaction_history)
    results.append({
        "transaction_id": txn.transaction_id,
        "actual_fraud": int(row['isFraud']),
        "predicted_decision": eval_result["decision"],
        "total_score": eval_result["total_score"],
    })

results_df = pd.DataFrame(results)
results_df.head()

,transaction_id,actual_fraud,predicted_decision,total_score
0,2987000,0,APPROVE,0.0
1,2987001,0,APPROVE,0.0
2,2987002,0,APPROVE,0.0
3,2987003,0,APPROVE,0.0
4,2987004,0,APPROVE,0.0


In [41]:
pd.crosstab(results_df['actual_fraud'], results_df['predicted_decision'])

predicted_decision,APPROVE
actual_fraud,
0,4891
1,109


In [42]:
df[df['isFraud'] == 1]['TransactionAmt'].describe()

count     109.000000
mean      182.921651
std       276.714486
min         5.000000
25%        58.004000
50%       100.000000
75%       200.000000
max      2161.000000
Name: TransactionAmt, dtype: float64

In [43]:
df[df['isFraud'] == 0]['TransactionAmt'].describe()

count    4891.000000
mean      144.618651
std       249.228880
min         1.896000
25%        47.950000
50%        77.950000
75%       150.000000
max      3247.910000
Name: TransactionAmt, dtype: float64

In [44]:
df['card1'].value_counts().head(10)

7919     571
9500     125
17188     73
15885     72
12544     70
2803      60
12695     47
18132     47
12839     45
15066     43
Name: card1, dtype: int64

In [45]:
df[df['isFraud'] == 1]['TransactionAmt'].describe()

count     109.000000
mean      182.921651
std       276.714486
min         5.000000
25%        58.004000
50%       100.000000
75%       200.000000
max      2161.000000
Name: TransactionAmt, dtype: float64

In [46]:
df[df['isFraud'] == 0]['TransactionAmt'].describe()

count    4891.000000
mean      144.618651
std       249.228880
min         1.896000
25%        47.950000
50%        77.950000
75%       150.000000
max      3247.910000
Name: TransactionAmt, dtype: float64

In [47]:
df[df['isFraud'] == 1]['TransactionAmt'].describe()

count     109.000000
mean      182.921651
std       276.714486
min         5.000000
25%        58.004000
50%       100.000000
75%       200.000000
max      2161.000000
Name: TransactionAmt, dtype: float64

In [48]:
df.groupby('isFraud')['card1'].apply(lambda x: x.value_counts().head(3))

isFraud       
0        7919     569
         9500     122
         17188     72
1        4504       6
         8732       5
         7585       3
Name: card1, dtype: int64

In [49]:
df.groupby('isFraud').apply(lambda g: (g['P_emaildomain'] != g['R_emaildomain']).mean())

isFraud
0    0.888980
1    0.623853
dtype: float64

In [51]:
df['R_emaildomain'].isna().mean()

0.8474

In [52]:
df['P_emaildomain'].isna().mean()

0.2212

In [54]:
df['P_emaildomain'].isna().mean()

0.2212

In [55]:
both_present = df[df['P_emaildomain'].notna() & df['R_emaildomain'].notna()]

both_present.groupby('isFraud').apply(
    lambda g: (g['P_emaildomain'] != g['R_emaildomain']).mean()
)

isFraud
0    0.115635
1    0.000000
dtype: float64

In [56]:
for col in ['C1', 'C2', 'C13', 'C14']:
    print(col)
    print(df.groupby('isFraud')[col].mean())
    print()

C1
isFraud
0    9.151912
1    7.266055
Name: C1, dtype: float64

C2
isFraud
0    8.722961
1    8.899083
Name: C2, dtype: float64

C13
isFraud
0    30.658557
1    10.275229
Name: C13, dtype: float64

C14
isFraud
0    6.953997
1    2.009174
Name: C14, dtype: float64



In [58]:
def check_low_c14(txn_row, threshold=3.0):
    triggered = txn_row['C14']<threshold;
    reason = f"C14={txn_row['C14']} below threshold {threshold}" if triggered else ""        

    return {
        "rule_name": "low_c14",
        "triggered": triggered,
        "score": 2.0 if triggered else 0.0,
        "reason": reason,
    }

In [59]:
row = df.iloc[0]
check_low_c14(row)

{'rule_name': 'low_c14',
 'triggered': True,
 'score': 2.0,
 'reason': 'C14=1.0 below threshold 3.0'}

In [61]:
@dataclass
class Transaction:
    transaction_id: str
    card_id: str
    amount: float
    currency: str
    merchant_id: str
    mcc: str
    country: str
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    channel: str = "card_present"
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    card_country: Optional[str] = None
    device_id: Optional[str] = None   # <-- new field

In [62]:
device_history = defaultdict(list)  # device_id -> list of past Transactions

In [64]:
def check_device_sharing(txn, device_history, window_hours=24, max_distinct_cards=3):
    if txn.device_id is None:
        return {"rule_name": "device_sharing", "triggered": False, "score": 0.0, "reason": "no device_id"}

    window = timedelta(hours=window_hours)
    cutoff = txn.timestamp - window

    recent = [t for t in device_history[txn.device_id] if t.timestamp >= cutoff]
    distinct_cards = {t.card_id for t in recent} | {txn.card_id}
    triggered = len(distinct_cards) > max_distinct_cards
    reason = f"{len(distinct_cards)} distinct cards on device {txn.device_id} in {window_hours}h (limit {max_distinct_cards})" if triggered else ""
    return {
        "rule_name": "device_sharing",
        "triggered": triggered,
        "score": 2.5 if triggered else 0.0,
        "reason": reason,
    }

In [65]:
now3 = datetime.now(timezone.utc)

for i in range(4):
    t = Transaction(
        transaction_id=f"d{i}", card_id=f"card_{i}", amount=30.00, currency="USD",
        merchant_id="m_x", mcc="5812", country="US",
        timestamp=now3 + timedelta(minutes=i * 5),
        device_id="device_shared_1",
    )
    result = check_device_sharing(t, device_history)
    print(f"card_{i}: triggered={result['triggered']}  {result['reason']}")
    device_history[t.device_id].append(t)

card_0: triggered=False  
card_1: triggered=False  
card_2: triggered=False  
card_3: triggered=True  4 distinct cards on device device_shared_1 in 24h (limit 3)


In [66]:
def check_blacklist(txn, blacklisted_cards):
    triggered = txn.card_id in blacklisted_cards
    reason = f"card {txn.card_id} is on the blacklist" if triggered else ""

    return {
        "rule_name": "blacklist",
        "triggered": triggered,
        "score": 10.0 if triggered else 0.0,  # high enough to force DECLINE alone
        "reason": reason,
    }

In [67]:
blacklisted_cards = {"card_stolen_001", "card_stolen_002"}

bad_txn = Transaction(
    transaction_id="bl1", card_id="card_stolen_001", amount=25.00, currency="USD",
    merchant_id="m_x", mcc="5812", country="US",
)

check_blacklist(bad_txn, blacklisted_cards)

{'rule_name': 'blacklist',
 'triggered': True,
 'score': 10.0,
 'reason': 'card card_stolen_001 is on the blacklist'}

In [68]:
result = run_engine(bad_txn, transaction_history, rules=[
    lambda t, h: check_amount_threshold(t, h),
    lambda t, h: check_velocity(t, h),
    lambda t, h: check_blacklist(t, blacklisted_cards),
])
result

{'transaction_id': 'bl1',
 'total_score': 10.0,
 'decision': 'DECLINE',
 'triggered_rules': [{'rule_name': 'blacklist',
   'triggered': True,
   'score': 10.0,
   'reason': 'card card_stolen_001 is on the blacklist'}]}